In [1]:
import json
import numpy as np

def check(name, ref_value, student_value):
    ref_arr = np.array(ref_value, dtype=float)
    stu_arr = np.array(student_value, dtype=float)
    if ref_arr.shape != stu_arr.shape:
        print(f"FAIL  {name}")
        print(f"      Shape mismatch: expected {ref_arr.shape}, got {stu_arr.shape}")
        return False
    match = np.allclose(ref_arr, stu_arr, rtol=1e-3, atol=1e-3)
    if match:
        print(f"PASS  {name}")
    else:
        error = np.abs(ref_arr - stu_arr)
        idx   = np.unravel_index(np.argmax(error), error.shape)
        print(f"FAIL  {name}")
        print(f"      Max error at position {idx}")
        print(f"      Reference : {ref_arr[idx]:.6f}")
        print(f"      Your value: {stu_arr[idx]:.6f}")
    return match


def validate(student: dict, json_path: str):
    with open(json_path) as f:
        ref = json.load(f)

    print("=" * 50)
    print(f"  {ref['element']} Validation -- {ref['label']}")
    print("=" * 50)

    passed = 0
    total  = 0

    for key in ['nodes', 'area', 'B', 'N_centroid', 'K']:
        if key in student:
            total  += 1
            passed += check(key, ref[key], student[key])

    if 'gauss_points' in student:
        ref_gps = ref['gauss_points']
        stu_gps = student['gauss_points']
        if len(stu_gps) != len(ref_gps):
            print(f"FAIL  gauss_points -- expected {len(ref_gps)} points, got {len(stu_gps)}")
            total += 1
        else:
            coord_key = 'xi' if 'xi' in ref_gps[0] else 'zeta'
            for i, (ref_gp, stu_gp) in enumerate(zip(ref_gps, stu_gps)):
                c1 = ref_gp[coord_key]
                c2 = ref_gp['eta']
                print(f"\n  Gauss point {i+1}  ({coord_key}={c1:.4f}, eta={c2:.4f})")
                for key in ['N', 'J', 'detJ', 'B']:
                    if key in stu_gp:
                        total  += 1
                        passed += check(f"  gp[{i+1}].{key}", ref_gp[key], stu_gp[key])

    print("=" * 50)
    print(f"  {passed}/{total} passed")
    print("=" * 50)



In [6]:
# CST
results_CST = {
    'nodes': np.array([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0]]),
    'K': np.array([
        [ 1483516.48,  714285.71, -1098901.10, -384615.38, -384615.38,  -329670.33],
        [  714285.71, 1483516.48,  -329670.33, -384615.38, -384615.38, -1098901.10],
        [-1098901.10, -329670.33,  1098901.10,       0.00,       0.00,   329670.33],
        [ -384615.38, -384615.38,       0.00,   384615.38,  384615.38,       0.00],
        [ -384615.38, -384615.38,       0.00,   384615.38,  384615.38,       0.00],
        [ -329670.33,-1098901.10,   329670.33,       0.00,       0.00,  1098901.10],
    ]),
}
validate(results_CST, r'C:\Dropbox\01. Brain\11. GitHub\FEM\examples\validation\CST_regular.json')


# LST
results_LST = {
    'nodes': np.array([[0.0,0.0],[1.0,0.0],[0.0,1.0],[0.5,0.0],[0.5,0.5],[0.0,0.5]]),
    'K': np.array([
        [ 1483516.48,  714285.71,  366300.37,  128205.13,  128205.13,  109890.11, -1465201.47, -512820.51,        0.00,       0.00, -512820.51, -439560.44],
        [  714285.71, 1483516.48,  109890.11,  128205.13,  128205.13,  366300.37,  -439560.44, -512820.51,        0.00,       0.00, -512820.51,-1465201.47],
        [  366300.37,  109890.11, 1098901.10,       0.00,       0.00, -109890.11, -1465201.47, -439560.44,        0.00,  439560.44,       0.00,       0.00],
        [  128205.13,  128205.13,       0.00,  384615.38, -128205.13,       0.00,  -512820.51, -512820.51,   512820.51,       0.00,       0.00,       0.00],
        [  128205.13,  128205.13,       0.00, -128205.13,  384615.38,       0.00,        0.00,       0.00,        0.00,  512820.51, -512820.51, -512820.51],
        [  109890.11,  366300.37, -109890.11,       0.00,       0.00, 1098901.10,        0.00,       0.00,   439560.44,       0.00, -439560.44,-1465201.47],
        [-1465201.47, -439560.44,-1465201.47, -512820.51,       0.00,       0.00,  3956043.96,  952380.95, -1025641.03, -952380.95,       0.00,  952380.95],
        [ -512820.51, -512820.51, -439560.44, -512820.51,       0.00,       0.00,   952380.95, 3956043.96,  -952380.95,-2930402.93,  952380.95,       0.00],
        [       0.00,       0.00,       0.00,  512820.51,       0.00,  439560.44, -1025641.03, -952380.95,  3956043.96,  952380.95,-2930402.93, -952380.95],
        [       0.00,       0.00,  439560.44,       0.00,  512820.51,       0.00,  -952380.95,-2930402.93,   952380.95, 3956043.96, -952380.95,-1025641.03],
        [ -512820.51, -512820.51,       0.00,       0.00, -512820.51, -439560.44,        0.00,  952380.95, -2930402.93, -952380.95, 3956043.96,  952380.95],
        [ -439560.44,-1465201.47,       0.00,       0.00, -512820.51,-1465201.47,   952380.95,       0.00,  -952380.95,-1025641.03,  952380.95, 3956043.96],
    ]),
}
validate(results_LST, r'C:\Dropbox\01. Brain\11. GitHub\FEM\examples\validation\LST_regular.json')


# Quad4
results_Quad4 = {
    'nodes': np.array([[0.0,0.0],[1.0,0.0],[1.0,1.0],[0.0,1.0]]),
    'K': np.array([
        [ 989010.99,  357142.86, -604395.60,  -27472.53, -494505.49, -357142.86,  109890.11,   27472.53],
        [ 357142.86,  989010.99,   27472.53,  109890.11, -357142.86, -494505.49,  -27472.53, -604395.60],
        [-604395.60,   27472.53,  989010.99, -357142.86,  109890.11,  -27472.53, -494505.49,  357142.86],
        [ -27472.53,  109890.11, -357142.86,  989010.99,   27472.53, -604395.60,  357142.86, -494505.49],
        [-494505.49, -357142.86,  109890.11,   27472.53,  989010.99,  357142.86, -604395.60,  -27472.53],
        [-357142.86, -494505.49,  -27472.53, -604395.60,  357142.86,  989010.99,   27472.53,  109890.11],
        [ 109890.11,  -27472.53, -494505.49,  357142.86, -604395.60,   27472.53,  989010.99, -357142.86],
        [  27472.53, -604395.60,  357142.86, -494505.49,  -27472.53,  109890.11, -357142.86,  989010.99],
    ]),
}
validate(results_Quad4, r'C:\Dropbox\01. Brain\11. GitHub\FEM\examples\validation\Quad4_regular.json')


# Quad9
results_Quad9 = {
    'nodes': np.array([[0.0,0.0],[1.0,0.0],[1.0,1.0],[0.0,1.0],
                       [0.5,0.0],[1.0,0.5],[0.5,1.0],[0.0,0.5],[0.5,0.5]]),
    'K': np.array([
        [ 923076.92,  357142.86,   37851.04,    9157.51,  -32967.03,  -39682.54, -136752.14,   -9157.51, -661782.66,  -36630.04,  117216.12,  158730.16,  212454.21,  158730.16,   68376.07,   36630.04, -527472.53, -634920.63],
        [ 357142.86,  923076.92,   -9157.51, -136752.14,  -39682.54,  -32967.03,   9157.51,   37851.04,   36630.04,   68376.07,  158730.16,  212454.21,  158730.16,  117216.12,  -36630.04, -661782.66, -634920.63, -527472.53],
        [  37851.04,   -9157.51,  923076.92, -357142.86, -136752.14,    9157.51,  -32967.03,   39682.54, -661782.66,   36630.04,   68376.07,  -36630.04,  212454.21, -158730.16,  117216.12, -158730.16, -527472.53,  634920.63],
        [   9157.51, -136752.14, -357142.86,  923076.92,   -9157.51,   37851.04,   39682.54,  -32967.03,  -36630.04,   68376.07,   36630.04, -661782.66, -158730.16,  117216.12, -158730.16,  212454.21,  634920.63, -527472.53],
        [ -32967.03,  -39682.54, -136752.14,   -9157.51,  923076.92,  357142.86,   37851.04,    9157.51,  212454.21,  158730.16,   68376.07,   36630.04, -661782.66,  -36630.04,  117216.12,  158730.16, -527472.53, -634920.63],
        [ -39682.54,  -32967.03,    9157.51,   37851.04,  357142.86,  923076.92,   -9157.51, -136752.14,  158730.16,  117216.12,  -36630.04, -661782.66,   36630.04,   68376.07,  158730.16,  212454.21, -634920.63, -527472.53],
        [-136752.14,    9157.51,  -32967.03,   39682.54,   37851.04,   -9157.51,  923076.92, -357142.86,  212454.21, -158730.16,  117216.12, -158730.16, -661782.66,   36630.04,   68376.07,  -36630.04, -527472.53,  634920.63],
        [  -9157.51,   37851.04,   39682.54,  -32967.03,    9157.51, -136752.14, -357142.86,  923076.92, -158730.16,  117216.12, -158730.16,  212454.21,  -36630.04,   68376.07,   36630.04, -661782.66,  634920.63, -527472.53],
        [-661782.66,   36630.04, -661782.66,  -36630.04,  212454.21,  158730.16,  212454.21, -158730.16, 2520146.52,       0.00, -527472.53, -634920.63, -253968.25,       0.00, -527472.53,  634920.63, -312576.31,       0.00],
        [ -36630.04,   68376.07,   36630.04,   68376.07,  158730.16,  117216.12, -158730.16,  117216.12,       0.00, 3282051.28, -634920.63, -527472.53,       0.00,  253968.25,  634920.63, -527472.53,       0.00,-2852258.85],
        [ 117216.12,  158730.16,   68376.07,   36630.04,   68376.07,  -36630.04,  117216.12, -158730.16, -527472.53, -634920.63, 3282051.28,       0.00, -527472.53,  634920.63,  253968.25,       0.00,-2852258.85,       0.00],
        [ 158730.16,  212454.21,  -36630.04, -661782.66,   36630.04, -661782.66, -158730.16,  212454.21, -634920.63, -527472.53,       0.00, 2520146.52,  634920.63, -527472.53,       0.00, -253968.25,       0.00, -312576.31],
        [ 212454.21,  158730.16,  212454.21, -158730.16, -661782.66,   36630.04, -661782.66,  -36630.04, -253968.25,       0.00, -527472.53,  634920.63, 2520146.52,       0.00, -527472.53, -634920.63, -312576.31,       0.00],
        [ 158730.16,  117216.12, -158730.16,  117216.12,  -36630.04,   68376.07,   36630.04,   68376.07,       0.00,  253968.25,  634920.63, -527472.53,       0.00, 3282051.28, -634920.63, -527472.53,       0.00,-2852258.85],
        [  68376.07,  -36630.04,  117216.12, -158730.16,  117216.12,  158730.16,   68376.07,   36630.04, -527472.53,  634920.63,  253968.25,       0.00, -527472.53, -634920.63, 3282051.28,       0.00,-2852258.85,       0.00],
        [  36630.04, -661782.66, -158730.16,  212454.21,  158730.16,  212454.21,  -36630.04, -661782.66,  634920.63, -527472.53,       0.00, -253968.25, -634920.63, -527472.53,       0.00, 2520146.52,       0.00, -312576.31],
        [-527472.53, -634920.63, -527472.53,  634920.63, -527472.53, -634920.63, -527472.53,  634920.63, -312576.31,       0.00,-2852258.85,       0.00, -312576.31,       0.00,-2852258.85,       0.00, 8439560.44,       0.00],
        [-634920.63, -527472.53,  634920.63, -527472.53, -634920.63, -527472.53,  634920.63, -527472.53,       0.00,-2852258.85,       0.00, -312576.31,       0.00,-2852258.85,       0.00, -312576.31,       0.00, 8439560.44],
    ]),
}
validate(results_Quad9, r'C:\Dropbox\01. Brain\11. GitHub\FEM\examples\validation\Quad9_regular.json')

  CST Validation -- CST_regular
PASS  nodes
PASS  K
  2/2 passed
  LST Validation -- LST_regular
PASS  nodes
PASS  K
  2/2 passed
  Quad4 Validation -- Quad4_regular
PASS  nodes
PASS  K
  2/2 passed
  Quad9 Validation -- Quad9_regular
PASS  nodes
PASS  K
  2/2 passed


In [2]:

# CST -- nodes [[0,0],[1,0],[0,1]], E=200000, nu=0.3, t=10
results_CST = {
    'nodes': np.array([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0]]),
    'area': 0.5,
    'B': np.array([
        [-1.0,  0.0,  1.0,  0.0,  0.0,  0.0],
        [ 0.0, -1.0,  0.0,  0.0,  0.0,  1.0],
        [-1.0, -1.0,  0.0,  1.0,  1.0,  0.0],
    ]),
    'N_centroid': np.array([
        [ 1.0,  0.0, -0.6667,  0.0, -0.6667,  0.0],
        [ 0.0,  1.0,  0.0,   -0.6667,  0.0,  -0.6667],
    ]),
    'K': np.array([
        [ 1483516.48,  714285.71, -1098901.10, -384615.38, -384615.38,  -329670.33],
        [  714285.71, 1483516.48,  -329670.33, -384615.38, -384615.38, -1098901.10],
        [-1098901.10, -329670.33,  1098901.10,       0.00,       0.00,   329670.33],
        [ -384615.38, -384615.38,       0.00,   384615.38,  384615.38,       0.00],
        [ -384615.38, -384615.38,       0.00,   384615.38,  384615.38,       0.00],
        [ -329670.33,-1098901.10,   329670.33,       0.00,       0.00,  1098901.10],
    ]),
}
validate(results_CST, r'C:\Dropbox\01. Brain\11. GitHub\FEM\examples\validation\CST_regular.json')



  CST Validation -- CST_regular
PASS  nodes
PASS  area
PASS  B
PASS  N_centroid
PASS  K
  5/5 passed


In [3]:

# Quad4 -- nodes [[0,0],[1,0],[1,1],[0,1]], E=200000, nu=0.3, t=10
# 9 Gauss points (3x3), B(3x8), N(2x8), J(2x2)
results_Quad4 = {
    'nodes': np.array([[0.0, 0.0], [1.0, 0.0], [1.0, 1.0], [0.0, 1.0]]),
    'area': 1.0,
    'K': np.array([
        [ 989010.99,  357142.86, -604395.60,  -27472.53, -494505.49, -357142.86,  109890.11,   27472.53],
        [ 357142.86,  989010.99,   27472.53,  109890.11, -357142.86, -494505.49,  -27472.53, -604395.60],
        [-604395.60,   27472.53,  989010.99, -357142.86,  109890.11,  -27472.53, -494505.49,  357142.86],
        [ -27472.53,  109890.11, -357142.86,  989010.99,   27472.53, -604395.60,  357142.86, -494505.49],
        [-494505.49, -357142.86,  109890.11,   27472.53,  989010.99,  357142.86, -604395.60,  -27472.53],
        [-357142.86, -494505.49,  -27472.53, -604395.60,  357142.86,  989010.99,   27472.53,  109890.11],
        [ 109890.11,  -27472.53, -494505.49,  357142.86, -604395.60,   27472.53,  989010.99, -357142.86],
        [  27472.53, -604395.60,  357142.86, -494505.49,  -27472.53,  109890.11, -357142.86,  989010.99],
    ]),
    'gauss_points': [
        {'N'   : np.array([[0.7873, 0.0, 0.1000, 0.0, 0.0127, 0.0, 0.1000, 0.0],
                           [0.0, 0.7873, 0.0, 0.1000, 0.0, 0.0127, 0.0, 0.1000]]),
         'J'   : np.array([[0.5, 0.0], [0.0, 0.5]]),
         'detJ': 0.25,
         'B'   : np.array([[-0.8873, 0.0,  0.8873, 0.0,  0.1127, 0.0, -0.1127, 0.0],
                            [ 0.0, -0.8873, 0.0, -0.1127, 0.0,  0.1127, 0.0,  0.8873],
                            [-0.8873, -0.8873, -0.1127, 0.8873, 0.1127, 0.1127, 0.8873, -0.1127]])},
        {'N'   : np.array([[0.4436, 0.0, 0.0564, 0.0, 0.0564, 0.0, 0.4436, 0.0],
                           [0.0, 0.4436, 0.0, 0.0564, 0.0, 0.0564, 0.0, 0.4436]]),
         'J'   : np.array([[0.5, 0.0], [0.0, 0.5]]),
         'detJ': 0.25,
         'B'   : np.array([[-0.5000, 0.0,  0.5000, 0.0,  0.5000, 0.0, -0.5000, 0.0],
                            [ 0.0, -0.8873, 0.0, -0.1127, 0.0,  0.1127, 0.0,  0.8873],
                            [-0.8873, -0.5000, -0.1127, 0.5000, 0.1127, 0.5000, 0.8873, -0.5000]])},
        {'N'   : np.array([[0.1000, 0.0, 0.0127, 0.0, 0.1000, 0.0, 0.7873, 0.0],
                           [0.0, 0.1000, 0.0, 0.0127, 0.0, 0.1000, 0.0, 0.7873]]),
         'J'   : np.array([[0.5, 0.0], [0.0, 0.5]]),
         'detJ': 0.25,
         'B'   : np.array([[-0.1127, 0.0,  0.1127, 0.0,  0.8873, 0.0, -0.8873, 0.0],
                            [ 0.0, -0.8873, 0.0, -0.1127, 0.0,  0.1127, 0.0,  0.8873],
                            [-0.8873, -0.1127, -0.1127, 0.1127, 0.1127, 0.8873, 0.8873, -0.8873]])},
        {'N'   : np.array([[0.4436, 0.0, 0.4436, 0.0, 0.0564, 0.0, 0.0564, 0.0],
                           [0.0, 0.4436, 0.0, 0.4436, 0.0, 0.0564, 0.0, 0.0564]]),
         'J'   : np.array([[0.5, 0.0], [0.0, 0.5]]),
         'detJ': 0.25,
         'B'   : np.array([[-0.8873, 0.0,  0.8873, 0.0,  0.1127, 0.0, -0.1127, 0.0],
                            [ 0.0, -0.5000, 0.0, -0.5000, 0.0,  0.5000, 0.0,  0.5000],
                            [-0.5000, -0.8873, -0.5000, 0.8873, 0.5000, 0.1127, 0.5000, -0.1127]])},
        {'N'   : np.array([[0.25, 0.0, 0.25, 0.0, 0.25, 0.0, 0.25, 0.0],
                           [0.0, 0.25, 0.0, 0.25, 0.0, 0.25, 0.0, 0.25]]),
         'J'   : np.array([[0.5, 0.0], [0.0, 0.5]]),
         'detJ': 0.25,
         'B'   : np.array([[-0.5, 0.0,  0.5, 0.0,  0.5, 0.0, -0.5, 0.0],
                            [ 0.0, -0.5, 0.0, -0.5, 0.0,  0.5, 0.0,  0.5],
                            [-0.5, -0.5, -0.5, 0.5,  0.5,  0.5,  0.5, -0.5]])},
        {'N'   : np.array([[0.0564, 0.0, 0.0564, 0.0, 0.4436, 0.0, 0.4436, 0.0],
                           [0.0, 0.0564, 0.0, 0.0564, 0.0, 0.4436, 0.0, 0.4436]]),
         'J'   : np.array([[0.5, 0.0], [0.0, 0.5]]),
         'detJ': 0.25,
         'B'   : np.array([[-0.1127, 0.0,  0.1127, 0.0,  0.8873, 0.0, -0.8873, 0.0],
                            [ 0.0, -0.5000, 0.0, -0.5000, 0.0,  0.5000, 0.0,  0.5000],
                            [-0.5000, -0.1127, -0.5000, 0.1127, 0.5000, 0.8873, 0.5000, -0.8873]])},
        {'N'   : np.array([[0.1000, 0.0, 0.7873, 0.0, 0.1000, 0.0, 0.0127, 0.0],
                           [0.0, 0.1000, 0.0, 0.7873, 0.0, 0.1000, 0.0, 0.0127]]),
         'J'   : np.array([[0.5, 0.0], [0.0, 0.5]]),
         'detJ': 0.25,
         'B'   : np.array([[-0.8873, 0.0,  0.8873, 0.0,  0.1127, 0.0, -0.1127, 0.0],
                            [ 0.0, -0.1127, 0.0, -0.8873, 0.0,  0.8873, 0.0,  0.1127],
                            [-0.1127, -0.8873, -0.8873, 0.8873, 0.8873, 0.1127, 0.1127, -0.1127]])},
        {'N'   : np.array([[0.0564, 0.0, 0.4436, 0.0, 0.4436, 0.0, 0.0564, 0.0],
                           [0.0, 0.0564, 0.0, 0.4436, 0.0, 0.4436, 0.0, 0.0564]]),
         'J'   : np.array([[0.5, 0.0], [0.0, 0.5]]),
         'detJ': 0.25,
         'B'   : np.array([[-0.5000, 0.0,  0.5000, 0.0,  0.5000, 0.0, -0.5000, 0.0],
                            [ 0.0, -0.1127, 0.0, -0.8873, 0.0,  0.8873, 0.0,  0.1127],
                            [-0.1127, -0.5000, -0.8873, 0.5000, 0.8873, 0.5000, 0.1127, -0.5000]])},
        {'N'   : np.array([[0.0127, 0.0, 0.1000, 0.0, 0.7873, 0.0, 0.1000, 0.0],
                           [0.0, 0.0127, 0.0, 0.1000, 0.0, 0.7873, 0.0, 0.1000]]),
         'J'   : np.array([[0.5, 0.0], [0.0, 0.5]]),
         'detJ': 0.25,
         'B'   : np.array([[-0.1127, 0.0,  0.1127, 0.0,  0.8873, 0.0, -0.8873, 0.0],
                            [ 0.0, -0.1127, 0.0, -0.8873, 0.0,  0.8873, 0.0,  0.1127],
                            [-0.1127, -0.1127, -0.8873, 0.1127, 0.8873, 0.8873, 0.1127, -0.8873]])},
    ],
}
validate(results_Quad4, r'C:\Dropbox\01. Brain\11. GitHub\FEM\examples\validation\Quad4_regular.json')

  Quad4 Validation -- Quad4_regular
PASS  nodes
PASS  area
PASS  K

  Gauss point 1  (zeta=-0.7746, eta=-0.7746)
PASS    gp[1].N
PASS    gp[1].J
PASS    gp[1].detJ
PASS    gp[1].B

  Gauss point 2  (zeta=-0.7746, eta=0.0000)
PASS    gp[2].N
PASS    gp[2].J
PASS    gp[2].detJ
PASS    gp[2].B

  Gauss point 3  (zeta=-0.7746, eta=0.7746)
PASS    gp[3].N
PASS    gp[3].J
PASS    gp[3].detJ
PASS    gp[3].B

  Gauss point 4  (zeta=0.0000, eta=-0.7746)
PASS    gp[4].N
PASS    gp[4].J
PASS    gp[4].detJ
PASS    gp[4].B

  Gauss point 5  (zeta=0.0000, eta=0.0000)
PASS    gp[5].N
PASS    gp[5].J
PASS    gp[5].detJ
PASS    gp[5].B

  Gauss point 6  (zeta=0.0000, eta=0.7746)
PASS    gp[6].N
PASS    gp[6].J
PASS    gp[6].detJ
PASS    gp[6].B

  Gauss point 7  (zeta=0.7746, eta=-0.7746)
PASS    gp[7].N
PASS    gp[7].J
PASS    gp[7].detJ
PASS    gp[7].B

  Gauss point 8  (zeta=0.7746, eta=0.0000)
PASS    gp[8].N
PASS    gp[8].J
PASS    gp[8].detJ
PASS    gp[8].B

  Gauss point 9  (zeta=0.7746, eta=0.7

In [4]:
# Quad9 -- nodes [[0,0],[1,0],[1,1],[0,1],[0.5,0],[1,0.5],[0.5,1],[0,0.5],[0.5,0.5]]
# E=200000, nu=0.3, t=10
# 9 Gauss points (3x3), B(3x18), N(2x18), J(2x2), K(18x18)
results_Quad9 = {
    'nodes': np.array([[0.0,0.0],[1.0,0.0],[1.0,1.0],[0.0,1.0],
                       [0.5,0.0],[1.0,0.5],[0.5,1.0],[0.0,0.5],[0.5,0.5]]),
    'area': 1.0,
    'K': np.array([
        [ 923076.92,  357142.86,   37851.04,    9157.51,  -32967.03,  -39682.54, -136752.14,   -9157.51, -661782.66,  -36630.04,  117216.12,  158730.16,  212454.21,  158730.16,   68376.07,   36630.04, -527472.53, -634920.63],
        [ 357142.86,  923076.92,   -9157.51, -136752.14,  -39682.54,  -32967.03,   9157.51,   37851.04,   36630.04,   68376.07,  158730.16,  212454.21,  158730.16,  117216.12,  -36630.04, -661782.66, -634920.63, -527472.53],
        [  37851.04,   -9157.51,  923076.92, -357142.86, -136752.14,    9157.51,  -32967.03,   39682.54, -661782.66,   36630.04,   68376.07,  -36630.04,  212454.21, -158730.16,  117216.12, -158730.16, -527472.53,  634920.63],
        [   9157.51, -136752.14, -357142.86,  923076.92,   -9157.51,   37851.04,   39682.54,  -32967.03,  -36630.04,   68376.07,   36630.04, -661782.66, -158730.16,  117216.12, -158730.16,  212454.21,  634920.63, -527472.53],
        [ -32967.03,  -39682.54, -136752.14,   -9157.51,  923076.92,  357142.86,   37851.04,    9157.51,  212454.21,  158730.16,   68376.07,   36630.04, -661782.66,  -36630.04,  117216.12,  158730.16, -527472.53, -634920.63],
        [ -39682.54,  -32967.03,    9157.51,   37851.04,  357142.86,  923076.92,   -9157.51, -136752.14,  158730.16,  117216.12,  -36630.04, -661782.66,   36630.04,   68376.07,  158730.16,  212454.21, -634920.63, -527472.53],
        [-136752.14,    9157.51,  -32967.03,   39682.54,   37851.04,   -9157.51,  923076.92, -357142.86,  212454.21, -158730.16,  117216.12, -158730.16, -661782.66,   36630.04,   68376.07,  -36630.04, -527472.53,  634920.63],
        [  -9157.51,   37851.04,   39682.54,  -32967.03,    9157.51, -136752.14, -357142.86,  923076.92, -158730.16,  117216.12, -158730.16,  212454.21,  -36630.04,   68376.07,   36630.04, -661782.66,  634920.63, -527472.53],
        [-661782.66,   36630.04, -661782.66,  -36630.04,  212454.21,  158730.16,  212454.21, -158730.16, 2520146.52,       0.00, -527472.53, -634920.63, -253968.25,       0.00, -527472.53,  634920.63, -312576.31,       0.00],
        [ -36630.04,   68376.07,   36630.04,   68376.07,  158730.16,  117216.12, -158730.16,  117216.12,       0.00, 3282051.28, -634920.63, -527472.53,       0.00,  253968.25,  634920.63, -527472.53,       0.00,-2852258.85],
        [ 117216.12,  158730.16,   68376.07,   36630.04,   68376.07,  -36630.04,  117216.12, -158730.16, -527472.53, -634920.63, 3282051.28,       0.00, -527472.53,  634920.63,  253968.25,       0.00,-2852258.85,       0.00],
        [ 158730.16,  212454.21,  -36630.04, -661782.66,   36630.04, -661782.66, -158730.16,  212454.21, -634920.63, -527472.53,       0.00, 2520146.52,  634920.63, -527472.53,       0.00, -253968.25,       0.00, -312576.31],
        [ 212454.21,  158730.16,  212454.21, -158730.16, -661782.66,   36630.04, -661782.66,  -36630.04, -253968.25,       0.00, -527472.53,  634920.63, 2520146.52,       0.00, -527472.53, -634920.63, -312576.31,       0.00],
        [ 158730.16,  117216.12, -158730.16,  117216.12,  -36630.04,   68376.07,   36630.04,   68376.07,       0.00,  253968.25,  634920.63, -527472.53,       0.00, 3282051.28, -634920.63, -527472.53,       0.00,-2852258.85],
        [  68376.07,  -36630.04,  117216.12, -158730.16,  117216.12,  158730.16,   68376.07,   36630.04, -527472.53,  634920.63,  253968.25,       0.00, -527472.53, -634920.63, 3282051.28,       0.00,-2852258.85,       0.00],
        [  36630.04, -661782.66, -158730.16,  212454.21,  158730.16,  212454.21,  -36630.04, -661782.66,  634920.63, -527472.53,       0.00, -253968.25, -634920.63, -527472.53,       0.00, 2520146.52,       0.00, -312576.31],
        [-527472.53, -634920.63, -527472.53,  634920.63, -527472.53, -634920.63, -527472.53,  634920.63, -312576.31,       0.00,-2852258.85,       0.00, -312576.31,       0.00,-2852258.85,       0.00, 8439560.44,       0.00],
        [-634920.63, -527472.53,  634920.63, -527472.53, -634920.63, -527472.53,  634920.63, -527472.53,       0.00,-2852258.85,       0.00, -312576.31,       0.00,-2852258.85,       0.00, -312576.31,       0.00, 8439560.44],
    ]),
    'gauss_points': [
        # gp 1: zeta=-0.7746, eta=-0.7746
        {'N'   : np.array([[ 0.4724, 0.0, -0.0600, 0.0,  0.0076, 0.0, -0.0600, 0.0,  0.2749, 0.0, -0.0349, 0.0, -0.0349, 0.0,  0.2749, 0.0,  0.1600, 0.0],
                            [ 0.0,  0.4724, 0.0, -0.0600, 0.0,  0.0076, 0.0, -0.0600, 0.0,  0.2749, 0.0, -0.0349, 0.0, -0.0349, 0.0,  0.2749, 0.0,  0.1600]]),
         'J'   : np.array([[0.5, 0.0], [0.0, 0.5]]),
         'detJ': 0.25,
         'B'   : np.array([[-1.7521, 0.0, -0.3775, 0.0,  0.0479, 0.0,  0.2225, 0.0,  2.1295, 0.0, -0.2197, 0.0, -0.2705, 0.0, -1.0197, 0.0,  1.2394, 0.0],
                            [ 0.0, -1.7521, 0.0,  0.2225, 0.0,  0.0479, 0.0, -0.3775, 0.0, -1.0197, 0.0, -0.2705, 0.0, -0.2197, 0.0,  2.1295, 0.0,  1.2394],
                            [-1.7521, -1.7521,  0.2225, -0.3775,  0.0479,  0.0479, -0.3775,  0.2225, -1.0197,  2.1295, -0.2705, -0.2197, -0.2197, -0.2705,  2.1295, -1.0197,  1.2394,  1.2394]])},
        # gp 2: zeta=-0.7746, eta=0.0
        {'N'   : np.array([[ 0.0, 0.0, -0.0, 0.0, -0.0, 0.0,  0.0, 0.0,  0.0, 0.0, -0.0873, 0.0,  0.0, 0.0,  0.6873, 0.0,  0.4, 0.0],
                            [ 0.0,  0.0, 0.0, -0.0, 0.0, -0.0, 0.0,  0.0,  0.0, 0.0,  0.0, -0.0873,  0.0, 0.0,  0.0,  0.6873,  0.0,  0.4]]),
         'J'   : np.array([[0.5, 0.0], [0.0, 0.5]]),
         'detJ': 0.25,
         'B'   : np.array([[ 0.0, 0.0, -0.0, 0.0,  0.0, 0.0, -0.0, 0.0,  0.0, 0.0, -0.5492, 0.0,  0.0, 0.0, -2.5492, 0.0,  3.0984, 0.0],
                            [ 0.0, -0.6873, 0.0,  0.0873, 0.0, -0.0873, 0.0,  0.6873, 0.0, -0.4, 0.0,  0.0, 0.0,  0.4, 0.0,  0.0, 0.0, -0.0],
                            [-0.6873,  0.0,  0.0873, -0.0, -0.0873,  0.0,  0.6873, -0.0, -0.4,  0.0,  0.0, -0.5492,  0.4,  0.0,  0.0, -2.5492, -0.0,  3.0984]])},
        # gp 3: zeta=-0.7746, eta=0.7746
        {'N'   : np.array([[-0.0600, 0.0,  0.0076, 0.0, -0.0600, 0.0,  0.4724, 0.0, -0.0349, 0.0, -0.0349, 0.0,  0.2749, 0.0,  0.2749, 0.0,  0.1600, 0.0],
                            [ 0.0, -0.0600, 0.0,  0.0076, 0.0, -0.0600, 0.0,  0.4724, 0.0, -0.0349, 0.0, -0.0349, 0.0,  0.2749, 0.0,  0.2749, 0.0,  0.1600]]),
         'J'   : np.array([[0.5, 0.0], [0.0, 0.5]]),
         'detJ': 0.25,
         'B'   : np.array([[ 0.2225, 0.0,  0.0479, 0.0, -0.3775, 0.0, -1.7521, 0.0, -0.2705, 0.0, -0.2197, 0.0,  2.1295, 0.0, -1.0197, 0.0,  1.2394, 0.0],
                            [ 0.0,  0.3775, 0.0, -0.0479, 0.0, -0.2225, 0.0,  1.7521, 0.0,  0.2197, 0.0,  0.2705, 0.0,  1.0197, 0.0, -2.1295, 0.0, -1.2394],
                            [ 0.3775,  0.2225, -0.0479,  0.0479, -0.2225, -0.3775,  1.7521, -1.7521,  0.2197, -0.2705,  0.2705, -0.2197,  1.0197,  2.1295, -2.1295, -1.0197, -1.2394,  1.2394]])},
        # gp 4: zeta=0.0, eta=-0.7746
        {'N'   : np.array([[ 0.0, 0.0,  0.0, 0.0, -0.0, 0.0, -0.0, 0.0,  0.6873, 0.0,  0.0, 0.0, -0.0873, 0.0,  0.0, 0.0,  0.4, 0.0],
                            [ 0.0,  0.0, 0.0,  0.0, 0.0, -0.0, 0.0, -0.0,  0.0,  0.6873, 0.0,  0.0,  0.0, -0.0873,  0.0,  0.0,  0.0,  0.4]]),
         'J'   : np.array([[0.5, 0.0], [0.0, 0.5]]),
         'detJ': 0.25,
         'B'   : np.array([[-0.6873, 0.0,  0.6873, 0.0, -0.0873, 0.0,  0.0873, 0.0,  0.0, 0.0,  0.4, 0.0,  0.0, 0.0, -0.4, 0.0, -0.0, 0.0],
                            [ 0.0,  0.0, 0.0, -0.0, 0.0,  0.0, 0.0, -0.0, 0.0, -2.5492, 0.0,  0.0, 0.0, -0.5492, 0.0,  0.0, 0.0,  3.0984],
                            [ 0.0, -0.6873, -0.0,  0.6873,  0.0, -0.0873, -0.0,  0.0873, -2.5492,  0.0,  0.0,  0.4, -0.5492,  0.0,  0.0, -0.4,  3.0984, -0.0]])},
        # gp 5: zeta=0.0, eta=0.0
        {'N'   : np.array([[ 0.0, 0.0,  0.0, 0.0,  0.0, 0.0,  0.0, 0.0,  0.0, 0.0,  0.0, 0.0,  0.0, 0.0,  0.0, 0.0,  1.0, 0.0],
                            [ 0.0,  0.0, 0.0,  0.0, 0.0,  0.0, 0.0,  0.0, 0.0,  0.0, 0.0,  0.0, 0.0,  0.0, 0.0,  0.0,  0.0,  1.0]]),
         'J'   : np.array([[0.5, 0.0], [0.0, 0.5]]),
         'detJ': 0.25,
         'B'   : np.array([[-0.0, 0.0,  0.0, 0.0,  0.0, 0.0, -0.0, 0.0,  0.0, 0.0,  1.0, 0.0, -0.0, 0.0, -1.0, 0.0, -0.0, 0.0],
                            [ 0.0,  0.0, 0.0, -0.0, 0.0,  0.0, 0.0,  0.0, 0.0, -1.0, 0.0, -0.0, 0.0,  1.0, 0.0,  0.0, 0.0,  0.0],
                            [ 0.0, -0.0, -0.0,  0.0,  0.0,  0.0,  0.0, -0.0, -1.0,  0.0, -0.0,  1.0,  1.0, -0.0,  0.0, -1.0,  0.0, -0.0]])},
        # gp 6: zeta=0.0, eta=0.7746
        {'N'   : np.array([[-0.0, 0.0, -0.0, 0.0,  0.0, 0.0,  0.0, 0.0, -0.0873, 0.0,  0.0, 0.0,  0.6873, 0.0,  0.0, 0.0,  0.4, 0.0],
                            [ 0.0, -0.0, 0.0, -0.0, 0.0,  0.0, 0.0,  0.0,  0.0, -0.0873,  0.0,  0.0,  0.0,  0.6873,  0.0,  0.0,  0.0,  0.4]]),
         'J'   : np.array([[0.5, 0.0], [0.0, 0.5]]),
         'detJ': 0.25,
         'B'   : np.array([[ 0.0873, 0.0, -0.0873, 0.0,  0.6873, 0.0, -0.6873, 0.0,  0.0, 0.0,  0.4, 0.0, -0.0, 0.0, -0.4, 0.0,  0.0, 0.0],
                            [ 0.0,  0.0, 0.0,  0.0, 0.0,  0.0, 0.0,  0.0, 0.0,  0.5492, 0.0, -0.0, 0.0,  2.5492, 0.0,  0.0, 0.0, -3.0984],
                            [ 0.0,  0.0873,  0.0, -0.0873,  0.0,  0.6873,  0.0, -0.6873,  0.5492,  0.0, -0.0,  0.4,  2.5492, -0.0,  0.0, -0.4, -3.0984,  0.0]])},
        # gp 7: zeta=0.7746, eta=-0.7746
        {'N'   : np.array([[-0.0600, 0.0,  0.4724, 0.0, -0.0600, 0.0,  0.0076, 0.0,  0.2749, 0.0,  0.2749, 0.0, -0.0349, 0.0, -0.0349, 0.0,  0.1600, 0.0],
                            [ 0.0, -0.0600, 0.0,  0.4724, 0.0, -0.0600, 0.0,  0.0076, 0.0,  0.2749, 0.0,  0.2749, 0.0, -0.0349, 0.0, -0.0349, 0.0,  0.1600]]),
         'J'   : np.array([[0.5, 0.0], [0.0, 0.5]]),
         'detJ': 0.25,
         'B'   : np.array([[ 0.3775, 0.0,  1.7521, 0.0, -0.2225, 0.0, -0.0479, 0.0, -2.1295, 0.0,  1.0197, 0.0,  0.2705, 0.0,  0.2197, 0.0, -1.2394, 0.0],
                            [ 0.0,  0.2225, 0.0, -1.7521, 0.0, -0.3775, 0.0,  0.0479, 0.0, -1.0197, 0.0,  2.1295, 0.0, -0.2197, 0.0, -0.2705, 0.0,  1.2394],
                            [ 0.2225,  0.3775, -1.7521,  1.7521, -0.3775, -0.2225,  0.0479, -0.0479, -1.0197, -2.1295,  2.1295,  1.0197, -0.2197,  0.2705, -0.2705,  0.2197,  1.2394, -1.2394]])},
        # gp 8: zeta=0.7746, eta=0.0
        {'N'   : np.array([[-0.0, 0.0,  0.0, 0.0,  0.0, 0.0, -0.0, 0.0,  0.0, 0.0,  0.6873, 0.0,  0.0, 0.0, -0.0873, 0.0,  0.4, 0.0],
                            [ 0.0, -0.0, 0.0,  0.0, 0.0,  0.0, 0.0, -0.0, 0.0,  0.0,  0.0,  0.6873,  0.0,  0.0,  0.0, -0.0873,  0.0,  0.4]]),
         'J'   : np.array([[0.5, 0.0], [0.0, 0.5]]),
         'detJ': 0.25,
         'B'   : np.array([[ 0.0, 0.0,  0.0, 0.0,  0.0, 0.0,  0.0, 0.0,  0.0, 0.0,  2.5492, 0.0, -0.0, 0.0,  0.5492, 0.0, -3.0984, 0.0],
                            [ 0.0,  0.0873, 0.0, -0.6873, 0.0,  0.6873, 0.0, -0.0873, 0.0, -0.4, 0.0, -0.0, 0.0,  0.4, 0.0,  0.0, 0.0,  0.0],
                            [ 0.0873,  0.0, -0.6873,  0.0,  0.6873,  0.0, -0.0873,  0.0, -0.4,  0.0, -0.0,  2.5492,  0.4, -0.0,  0.0,  0.5492,  0.0, -3.0984]])},
        # gp 9: zeta=0.7746, eta=0.7746
        {'N'   : np.array([[ 0.0076, 0.0, -0.0600, 0.0,  0.4724, 0.0, -0.0600, 0.0, -0.0349, 0.0,  0.2749, 0.0,  0.2749, 0.0, -0.0349, 0.0,  0.1600, 0.0],
                            [ 0.0,  0.0076, 0.0, -0.0600, 0.0,  0.4724, 0.0, -0.0600, 0.0, -0.0349, 0.0,  0.2749, 0.0,  0.2749, 0.0, -0.0349, 0.0,  0.1600]]),
         'J'   : np.array([[0.5, 0.0], [0.0, 0.5]]),
         'detJ': 0.25,
         'B'   : np.array([[-0.0479, 0.0, -0.2225, 0.0,  1.7521, 0.0,  0.3775, 0.0,  0.2705, 0.0,  1.0197, 0.0, -2.1295, 0.0,  0.2197, 0.0, -1.2394, 0.0],
                            [ 0.0, -0.0479, 0.0,  0.3775, 0.0,  1.7521, 0.0, -0.2225, 0.0,  0.2197, 0.0, -2.1295, 0.0,  1.0197, 0.0,  0.2705, 0.0, -1.2394],
                            [-0.0479, -0.0479,  0.3775, -0.2225,  1.7521,  1.7521, -0.2225,  0.3775,  0.2197,  0.2705, -2.1295,  1.0197,  1.0197, -2.1295,  0.2705,  0.2197, -1.2394, -1.2394]])},
    ],
}
validate(results_Quad9, r'C:\Dropbox\01. Brain\11. GitHub\FEM\examples\validation\Quad9_regular.json')



  Quad9 Validation -- Quad9_regular
PASS  nodes
PASS  area
PASS  K

  Gauss point 1  (zeta=-0.7746, eta=-0.7746)
PASS    gp[1].N
PASS    gp[1].J
PASS    gp[1].detJ
PASS    gp[1].B

  Gauss point 2  (zeta=-0.7746, eta=0.0000)
PASS    gp[2].N
PASS    gp[2].J
PASS    gp[2].detJ
PASS    gp[2].B

  Gauss point 3  (zeta=-0.7746, eta=0.7746)
PASS    gp[3].N
PASS    gp[3].J
PASS    gp[3].detJ
PASS    gp[3].B

  Gauss point 4  (zeta=0.0000, eta=-0.7746)
PASS    gp[4].N
PASS    gp[4].J
PASS    gp[4].detJ
PASS    gp[4].B

  Gauss point 5  (zeta=0.0000, eta=0.0000)
PASS    gp[5].N
PASS    gp[5].J
PASS    gp[5].detJ
PASS    gp[5].B

  Gauss point 6  (zeta=0.0000, eta=0.7746)
PASS    gp[6].N
PASS    gp[6].J
PASS    gp[6].detJ
PASS    gp[6].B

  Gauss point 7  (zeta=0.7746, eta=-0.7746)
PASS    gp[7].N
PASS    gp[7].J
PASS    gp[7].detJ
PASS    gp[7].B

  Gauss point 8  (zeta=0.7746, eta=0.0000)
PASS    gp[8].N
PASS    gp[8].J
PASS    gp[8].detJ
PASS    gp[8].B

  Gauss point 9  (zeta=0.7746, eta=0.7

In [5]:

# LST -- nodes [[0,0],[1,0],[0,1],[0.5,0],[0.5,0.5],[0,0.5]]
# E=200000, nu=0.3, t=10
# 9 Gauss points, B(3x12), N(2x12), J(2x2), K(12x12)
results_LST = {
    'nodes': np.array([[0.0,0.0],[1.0,0.0],[0.0,1.0],[0.5,0.0],[0.5,0.5],[0.0,0.5]]),
    'area': 0.5,
    'K': np.array([
        [ 1483516.48,  714285.71,  366300.37,  128205.13,  128205.13,  109890.11, -1465201.47, -512820.51,        0.00,       0.00, -512820.51, -439560.44],
        [  714285.71, 1483516.48,  109890.11,  128205.13,  128205.13,  366300.37,  -439560.44, -512820.51,        0.00,       0.00, -512820.51,-1465201.47],
        [  366300.37,  109890.11, 1098901.10,       0.00,       0.00, -109890.11, -1465201.47, -439560.44,        0.00,  439560.44,       0.00,       0.00],
        [  128205.13,  128205.13,       0.00,  384615.38, -128205.13,       0.00,  -512820.51, -512820.51,   512820.51,       0.00,       0.00,       0.00],
        [  128205.13,  128205.13,       0.00, -128205.13,  384615.38,       0.00,        0.00,       0.00,        0.00,  512820.51, -512820.51, -512820.51],
        [  109890.11,  366300.37, -109890.11,       0.00,       0.00, 1098901.10,        0.00,       0.00,   439560.44,       0.00, -439560.44,-1465201.47],
        [-1465201.47, -439560.44,-1465201.47, -512820.51,       0.00,       0.00,  3956043.96,  952380.95, -1025641.03, -952380.95,       0.00,  952380.95],
        [ -512820.51, -512820.51, -439560.44, -512820.51,       0.00,       0.00,   952380.95, 3956043.96,  -952380.95,-2930402.93,  952380.95,       0.00],
        [       0.00,       0.00,       0.00,  512820.51,       0.00,  439560.44, -1025641.03, -952380.95,  3956043.96,  952380.95,-2930402.93, -952380.95],
        [       0.00,       0.00,  439560.44,       0.00,  512820.51,       0.00,  -952380.95,-2930402.93,   952380.95, 3956043.96, -952380.95,-1025641.03],
        [ -512820.51, -512820.51,       0.00,       0.00, -512820.51, -439560.44,        0.00,  952380.95, -2930402.93, -952380.95, 3956043.96,  952380.95],
        [ -439560.44,-1465201.47,       0.00,       0.00, -512820.51,-1465201.47,   952380.95,       0.00,  -952380.95,-1025641.03,  952380.95, 3956043.96],
    ]),
    'gauss_points': [
        # gp 1: xi=0.1127, eta=0.1000
        {'N'   : np.array([[ 0.4524, 0.0, -0.0873, 0.0, -0.0800, 0.0,  0.3549, 0.0,  0.0451, 0.0,  0.3149, 0.0],
                            [ 0.0,  0.4524, 0.0, -0.0873, 0.0, -0.0800, 0.0,  0.3549, 0.0,  0.0451, 0.0,  0.3149]]),
         'J'   : np.array([[1.0, 0.0], [0.0, 1.0]]),
         'detJ': 1.0,
         'B'   : np.array([[-2.1492, 0.0, -0.5492, 0.0,  0.0, 0.0,  2.6984, 0.0,  0.4, 0.0, -0.4, 0.0],
                            [ 0.0, -2.1492, 0.0,  0.0, 0.0, -0.6, 0.0, -0.4508, 0.0,  0.4508, 0.0,  2.7492],
                            [-2.1492, -2.1492,  0.0, -0.5492, -0.6,  0.0, -0.4508,  2.6984,  0.4508,  0.4,  2.7492, -0.4]])},
        # gp 2: xi=0.1127, eta=0.4436
        {'N'   : np.array([[-0.0500, 0.0, -0.0873, 0.0, -0.0500, 0.0,  0.2000, 0.0,  0.2000, 0.0,  0.7873, 0.0],
                            [ 0.0, -0.0500, 0.0, -0.0873, 0.0, -0.0500, 0.0,  0.2000, 0.0,  0.2000, 0.0,  0.7873]]),
         'J'   : np.array([[1.0, 0.0], [0.0, 1.0]]),
         'detJ': 1.0,
         'B'   : np.array([[-0.7746, 0.0, -0.5492, 0.0,  0.0, 0.0,  1.3238, 0.0,  1.7746, 0.0, -1.7746, 0.0],
                            [ 0.0, -0.7746, 0.0,  0.0, 0.0,  0.7746, 0.0, -0.4508, 0.0,  0.4508, 0.0,  0.0],
                            [-0.7746, -0.7746,  0.0, -0.5492,  0.7746,  0.0, -0.4508,  1.3238,  0.4508,  1.7746,  0.0, -1.7746]])},
        # gp 3: xi=0.1127, eta=0.7873
        {'N'   : np.array([[-0.0800, 0.0, -0.0873, 0.0,  0.4524, 0.0,  0.0451, 0.0,  0.3549, 0.0,  0.3149, 0.0],
                            [ 0.0, -0.0800, 0.0, -0.0873, 0.0,  0.4524, 0.0,  0.0451, 0.0,  0.3549, 0.0,  0.3149]]),
         'J'   : np.array([[1.0, 0.0], [0.0, 1.0]]),
         'detJ': 1.0,
         'B'   : np.array([[ 0.6, 0.0, -0.5492, 0.0,  0.0, 0.0, -0.0508, 0.0,  3.1492, 0.0, -3.1492, 0.0],
                            [ 0.0,  0.6, 0.0,  0.0, 0.0,  2.1492, 0.0, -0.4508, 0.0,  0.4508, 0.0, -2.7492],
                            [ 0.6,  0.6,  0.0, -0.5492,  2.1492,  0.0, -0.4508, -0.0508,  0.4508,  3.1492, -2.7492, -3.1492]])},
        # gp 4: xi=0.5000, eta=0.0564
        {'N'   : np.array([[-0.0500, 0.0,  0.0, 0.0, -0.0500, 0.0,  0.8873, 0.0,  0.1127, 0.0,  0.1, 0.0],
                            [ 0.0, -0.0500, 0.0,  0.0, 0.0, -0.0500, 0.0,  0.8873, 0.0,  0.1127, 0.0,  0.1]]),
         'J'   : np.array([[1.0, 0.0], [0.0, 1.0]]),
         'detJ': 1.0,
         'B'   : np.array([[-0.7746, 0.0,  1.0, 0.0,  0.0, 0.0, -0.2254, 0.0,  0.2254, 0.0, -0.2254, 0.0],
                            [ 0.0, -0.7746, 0.0,  0.0, 0.0, -0.7746, 0.0, -2.0, 0.0,  2.0, 0.0,  1.5492],
                            [-0.7746, -0.7746,  0.0,  1.0, -0.7746,  0.0, -2.0, -0.2254,  2.0,  0.2254,  1.5492, -0.2254]])},
        # gp 5: xi=0.5000, eta=0.2500
        {'N'   : np.array([[-0.1250, 0.0,  0.0, 0.0, -0.1250, 0.0,  0.5, 0.0,  0.5, 0.0,  0.25, 0.0],
                            [ 0.0, -0.1250, 0.0,  0.0, 0.0, -0.1250, 0.0,  0.5, 0.0,  0.5, 0.0,  0.25]]),
         'J'   : np.array([[1.0, 0.0], [0.0, 1.0]]),
         'detJ': 1.0,
         'B'   : np.array([[-0.0, 0.0,  1.0, 0.0,  0.0, 0.0, -1.0, 0.0,  1.0, 0.0, -1.0, 0.0],
                            [ 0.0,  0.0, 0.0,  0.0, 0.0,  0.0, 0.0, -2.0, 0.0,  2.0, 0.0,  0.0],
                            [ 0.0, -0.0,  0.0,  1.0,  0.0,  0.0, -2.0, -1.0,  2.0,  1.0,  0.0, -1.0]])},
        # gp 6: xi=0.5000, eta=0.4436
        {'N'   : np.array([[-0.0500, 0.0,  0.0, 0.0, -0.0500, 0.0,  0.1127, 0.0,  0.8873, 0.0,  0.1, 0.0],
                            [ 0.0, -0.0500, 0.0,  0.0, 0.0, -0.0500, 0.0,  0.1127, 0.0,  0.8873, 0.0,  0.1]]),
         'J'   : np.array([[1.0, 0.0], [0.0, 1.0]]),
         'detJ': 1.0,
         'B'   : np.array([[ 0.7746, 0.0,  1.0, 0.0,  0.0, 0.0, -1.7746, 0.0,  1.7746, 0.0, -1.7746, 0.0],
                            [ 0.0,  0.7746, 0.0,  0.0, 0.0,  0.7746, 0.0, -2.0, 0.0,  2.0, 0.0, -1.5492],
                            [ 0.7746,  0.7746,  0.0,  1.0,  0.7746,  0.0, -2.0, -1.7746,  2.0,  1.7746, -1.5492, -1.7746]])},
        # gp 7: xi=0.8873, eta=0.0127
        {'N'   : np.array([[-0.0800, 0.0,  0.6873, 0.0, -0.0124, 0.0,  0.3549, 0.0,  0.0451, 0.0,  0.0051, 0.0],
                            [ 0.0, -0.0800, 0.0,  0.6873, 0.0, -0.0124, 0.0,  0.3549, 0.0,  0.0451, 0.0,  0.0051]]),
         'J'   : np.array([[1.0, 0.0], [0.0, 1.0]]),
         'detJ': 1.0,
         'B'   : np.array([[ 0.6, 0.0,  2.5492, 0.0,  0.0, 0.0, -3.1492, 0.0,  0.0508, 0.0, -0.0508, 0.0],
                            [ 0.0,  0.6, 0.0,  0.0, 0.0, -0.9492, 0.0, -3.5492, 0.0,  3.5492, 0.0,  0.3492],
                            [ 0.6,  0.6,  0.0,  2.5492, -0.9492,  0.0, -3.5492, -3.1492,  3.5492,  0.0508,  0.3492, -0.0508]])},
        # gp 8: xi=0.8873, eta=0.0564
        {'N'   : np.array([[-0.0500, 0.0,  0.6873, 0.0, -0.0500, 0.0,  0.2, 0.0,  0.2, 0.0,  0.0127, 0.0],
                            [ 0.0, -0.0500, 0.0,  0.6873, 0.0, -0.0500, 0.0,  0.2, 0.0,  0.2, 0.0,  0.0127]]),
         'J'   : np.array([[1.0, 0.0], [0.0, 1.0]]),
         'detJ': 1.0,
         'B'   : np.array([[ 0.7746, 0.0,  2.5492, 0.0,  0.0, 0.0, -3.3238, 0.0,  0.2254, 0.0, -0.2254, 0.0],
                            [ 0.0,  0.7746, 0.0,  0.0, 0.0, -0.7746, 0.0, -3.5492, 0.0,  3.5492, 0.0,  0.0],
                            [ 0.7746,  0.7746,  0.0,  2.5492, -0.7746,  0.0, -3.5492, -3.3238,  3.5492,  0.2254,  0.0, -0.2254]])},
        # gp 9: xi=0.8873, eta=0.1000
        {'N'   : np.array([[-0.0124, 0.0,  0.6873, 0.0, -0.0800, 0.0,  0.0451, 0.0,  0.3549, 0.0,  0.0051, 0.0],
                            [ 0.0, -0.0124, 0.0,  0.6873, 0.0, -0.0800, 0.0,  0.0451, 0.0,  0.3549, 0.0,  0.0051]]),
         'J'   : np.array([[1.0, 0.0], [0.0, 1.0]]),
         'detJ': 1.0,
         'B'   : np.array([[ 0.9492, 0.0,  2.5492, 0.0,  0.0, 0.0, -3.4984, 0.0,  0.4, 0.0, -0.4, 0.0],
                            [ 0.0,  0.9492, 0.0,  0.0, 0.0, -0.6, 0.0, -3.5492, 0.0,  3.5492, 0.0, -0.3492],
                            [ 0.9492,  0.9492,  0.0,  2.5492, -0.6,  0.0, -3.5492, -3.4984,  3.5492,  0.4, -0.3492, -0.4]])},
    ],
}
validate(results_LST, r'C:\Dropbox\01. Brain\11. GitHub\FEM\examples\validation\LST_regular.json')

  LST Validation -- LST_regular
PASS  nodes
PASS  area
PASS  K

  Gauss point 1  (xi=0.1127, eta=0.1000)
PASS    gp[1].N
PASS    gp[1].J
PASS    gp[1].detJ
PASS    gp[1].B

  Gauss point 2  (xi=0.1127, eta=0.4436)
PASS    gp[2].N
PASS    gp[2].J
PASS    gp[2].detJ
PASS    gp[2].B

  Gauss point 3  (xi=0.1127, eta=0.7873)
PASS    gp[3].N
PASS    gp[3].J
PASS    gp[3].detJ
PASS    gp[3].B

  Gauss point 4  (xi=0.5000, eta=0.0564)
PASS    gp[4].N
PASS    gp[4].J
PASS    gp[4].detJ
PASS    gp[4].B

  Gauss point 5  (xi=0.5000, eta=0.2500)
PASS    gp[5].N
PASS    gp[5].J
PASS    gp[5].detJ
PASS    gp[5].B

  Gauss point 6  (xi=0.5000, eta=0.4436)
PASS    gp[6].N
PASS    gp[6].J
PASS    gp[6].detJ
PASS    gp[6].B

  Gauss point 7  (xi=0.8873, eta=0.0127)
PASS    gp[7].N
PASS    gp[7].J
PASS    gp[7].detJ
PASS    gp[7].B

  Gauss point 8  (xi=0.8873, eta=0.0564)
PASS    gp[8].N
PASS    gp[8].J
PASS    gp[8].detJ
PASS    gp[8].B

  Gauss point 9  (xi=0.8873, eta=0.1000)
PASS    gp[9].N
PASS   